# Learned noise-aware attention — everything on Colab (Drive-resumable)

One notebook for the whole second contribution. **Every checkpoint, log and CSV lives in Google Drive**,
so Colab disconnects cannot lose work: when a session drops, just reconnect and **Run all again** — the
notebook skips finished steps and resumes unfinished ones.

**Stages** (set `STAGE` in cell 2):
- `'screen'`: 3 short runs (control, scale 0.5, scale 1.0), 20k iters each, **validation-only** selection. ~5 h.
- `'full'`: 150k control + selected variant, one after the other, then **one** held-out test evaluation. ~25 h total
  across several Colab sessions.

**Setup:** Runtime → Change runtime type → GPU (T4). Internet is on by default.

**Time reality:** a 150k run is ~12.5 h on one T4. Free Colab can drop a session every few hours, so the
`full` stage will need several reconnects over the two days — that is expected and handled. Put the notebook
in the same Google account whose Drive you mount.


In [ ]:
import os, sys, subprocess, glob, json, time, shutil
from pathlib import Path

# 1) Fresh code from GitHub; Drive holds all state/results.
if not Path('/content/btp').exists():
    !git clone -q https://github.com/Crazycraftie/low-light-enhancement-btp.git /content/btp
else:
    !cd /content/btp && git pull -q
os.chdir('/content/btp')

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE = Path('/content/drive/MyDrive/btp_learned_noise')
DATA_DIR = DRIVE / 'data'
RUNS = DRIVE / 'runs'
for d in (DATA_DIR, RUNS):
    d.mkdir(parents=True, exist_ok=True)

print('HEAD:', subprocess.check_output(['git','rev-parse','--short','HEAD'], text=True).strip())
print('Drive state:', DRIVE)


In [ ]:
# Auto-detect: no selection.json yet -> screen; once it exists -> full runs.
# Override by setting FORCE_STAGE = 'screen' or 'full' if you need to re-run a stage.
FORCE_STAGE = None
STAGE = FORCE_STAGE or ('full' if (DRIVE/'selection.json').exists() else 'screen')

subprocess.run([sys.executable,'-m','pip','install','-q','einops','gdown','lmdb','yacs','addict',
                'future','natsort','yapf','thop','timm','lpips','scikit-image'], check=True)
if not Path('baselines/Retinexformer').exists():
    !git clone -q --depth 1 https://github.com/caiyuanhao1998/Retinexformer.git baselines/Retinexformer
import torch
assert torch.cuda.is_available(), 'Enable GPU: Runtime -> Change runtime type -> T4'
GPU = torch.cuda.get_device_name(0)
OPT = 'baselines/Retinexformer/Options/RetinexFormer_LOL_v1.yml'
print('GPU:', GPU, '| stage:', STAGE)


## Shared helpers: data + weights (cached in Drive)


In [ ]:
def gdl(fid, out, tries=4):
    out = Path(out); out.parent.mkdir(parents=True, exist_ok=True)
    for _ in range(tries):
        if subprocess.run(['gdown','-q',fid,'-O',str(out)]).returncode==0 and out.exists(): return
    raise RuntimeError('download failed '+str(out))

DATA = DATA_DIR / 'LOLv2/Synthetic'
if not (DATA/'Train/Low').is_dir():
    gdl('1Ou9EljYZW8o5dbDCf9R34FS8Pd8kEp2U', str(DATA_DIR/'lolv2.zip'))
    subprocess.run(['unzip','-q','-o',str(DATA_DIR/'lolv2.zip'),'-d',str(DATA_DIR)], check=True)
assert len(list((DATA/'Train/Low').glob('*')))==900, 'expected 900 train pairs'
assert len(list((DATA/'Test/Low').glob('*')))==100, 'expected 100 test pairs'

WDIR = Path('baselines/Retinexformer/pretrained_weights')
W = WDIR/'LOL_v2_synthetic.pth'
if not W.exists():
    c = list(WDIR.glob('**/LOL_v2_synthetic.pth'))
    if not c:
        subprocess.run(['gdown','--folder','1ynK5hfQachzc8y96ZumhkPPDXzHJwaQV','-O',str(DATA_DIR/'rf_weights')],check=True)
        c = list((DATA_DIR/'rf_weights').glob('**/LOL_v2_synthetic.pth'))
    shutil.copyfile(str(c[0]), str(W))
W = str(W)
print('data + weights ready:', W)


## Stage 1 — screening (3 x 20k, validation-only selection)


In [ ]:
def train_one(out, scale, iters, lr, timeout, resume=True):
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    if (out/'metrics.json').exists():
        print('already complete:', out); return 0
    cmd = ['timeout', str(timeout), sys.executable, 'scripts/finetune.py',
           '--loss','l1','--iters',str(iters),'--lr',str(lr),
           '--batch','8','--crop','128','--seed','42',
           '--pretrained',W,'--opt',OPT,'--repo','baselines/Retinexformer',
           '--data',str(DATA/'Train'),'--device','cuda',
           '--val-fraction','0.1','--val-seed','2026','--val-every',str(min(2000,iters//10)),
           '--checkpoint-every','5000','--log-every','500','--out',str(out)]
    if scale is not None: cmd += ['--learned-noise','--noise-scale',str(scale)]
    if resume and (out/'latest_state.pth').exists(): cmd += ['--resume']
    with (out/'run.log').open('a') as log:
        rc = subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT).returncode
    print(out.name, 'exit', rc)
    if rc not in (0,124): raise RuntimeError((out/'run.log').read_text()[-3000:])
    return rc

if STAGE == 'screen':
    for name, scale in (('control',None),('learned_s0.5',0.5),('learned_s1.0',1.0)):
        train_one(RUNS/'screen'/name, scale, 20000, 2e-5, 9000)
    scores = {}
    for name,_ in (('control',None),('learned_s0.5',0.5),('learned_s1.0',1.0)):
        m = RUNS/'screen'/name/'metrics.json'
        if m.exists(): scores[name]=json.loads(m.read_text())['best_val_psnr']
    print('validation PSNR:', scores)
    best = max(scores, key=scores.get) if scores else None
    (DRIVE/'selection.json').write_text(json.dumps({'scores':scores,'best':best},indent=2))
    print('SELECTED ON VALIDATION:', best)


## Stage 2 — final 150k matched runs + one held-out test evaluation


In [ ]:
def infer(weights, inp, out, method, scale=None):
    cmd = [sys.executable,'scripts/retinexformer_infer.py','--repo','baselines/Retinexformer',
           '--opt',OPT,'--weights',str(weights),'--input',str(inp),'--output',str(out),
           '--method',method,'--device','cuda','--no-speed-log']
    if scale is not None: cmd += ['--learned-noise','--noise-scale',str(scale)]
    subprocess.run(cmd, check=True)

if STAGE == 'full':
    sel = json.loads((DRIVE/'selection.json').read_text())
    print('Selection from screening:', sel)
    best = sel['best']
    if best not in ('learned_s0.5','learned_s1.0'):
        raise RuntimeError('Control won screening on validation - stop here; no gain to test.')
    SCALE = 0.5 if best=='learned_s0.5' else 1.0

    # Sequential on one Colab GPU. Run all: it resumes unfinished ones after a disconnect.
    train_one(RUNS/'final'/'control', None, 150000, 2e-4, 40000)
    train_one(RUNS/'final'/'learned_noise', SCALE, 150000, 2e-4, 40000)

    done = all((RUNS/'final'/n/'metrics.json').exists() for n in ('control','learned_noise'))
    if done:
        for name, scale in (('control',None),('learned_noise',SCALE)):
            run = RUNS/'final'/name
            w = run/'best.pth'
            if not w.exists(): w = run/'final.pth'
            method = 'LearnedNoise-control' if scale is None else f'LearnedNoise-a{SCALE}'
            out = DRIVE/'results'/method/'LOLv2-syn'
            infer(w, DATA/'Test/Low', out, method, scale)
            for script, extra in (
                ('scripts/evaluate.py', ['--gt',str(DATA/'Test/Normal')]),
                ('scripts/eval_dark_regions.py', ['--gt',str(DATA/'Test/Normal'),'--low',str(DATA/'Test/Low')])):
                subprocess.run([sys.executable,script,'--pred',str(out),'--method',method,
                                '--dataset','LOLv2-syn']+extra, check=True)
        # Copy the resulting CSVs to Drive for download.
        for f in ('results.csv','dark_region.csv'):
            shutil.copyfile(f'results/{f}', str(DRIVE/f))
        print('TEST RESULTS:')
        print('\n'.join(l for l in (DRIVE/'results.csv').read_text().splitlines() if 'LearnedNoise' in l))
    else:
        print('Training still incomplete after this session - reconnect and Run all again.')
        print('Saved checkpoints in Drive:', sorted(str(p.relative_to(DRIVE)) for p in RUNS.rglob('metrics.json')))


## Done

Download `results.csv` and `dark_region.csv` from `MyDrive/btp_learned_noise/` and send them.
If a session dropped mid-run, just re-run this notebook — finished runs are skipped and unfinished
ones resume from their Drive checkpoints.
